# exp-20260913-03 — what actually limits each target

**Zero GPU. No training. Nothing in the pipeline changes.** Config, data loading and the frozen
`v1-keyword` labeller cells are copied verbatim from `baseline-v1.ipynb`, so the labels analysed
here are exactly the labels the baseline trained on.

**The question as originally posed does not survive contact with the arithmetic** — see section 3.
Read that before using any number here.


In [ ]:
# ================================================================== CONFIG — the only cell to edit
from __future__ import annotations
import glob, json, os, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODE = "full"          # "smoke" (minutes, proves it runs) | "full" (the baseline) | "submit" (inference only)

# ---- competition constants (rules.md) -----------------------------------------------------------
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)
SUBMISSION_NAME = "submission.csv"        # rules.md: hard requirement
KAGGLE_LIMIT_H  = 9.0                     # rules.md: CPU or GPU notebook <= 9 h
WORKING_LIMIT_H = 6.75                    # rules.md: 9 h minus 25% headroom

# ---- versioned artefacts (rules.md: cache is keyed by preprocessing version) --------------------
LABELLER_VERSION  = "v1-keyword"          # weak-label rules; bump when the labeller changes
PREPROC_VERSION   = "p1"                  # bump on ANY change to the DICOM -> tensor path
EXPERIMENT_ID     = "baseline-v1"

# ---- study -> tensor geometry -------------------------------------------------------------------
SLOTS       = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]   # (plane, prefer fluid-sensitive)
N_SLOT      = len(SLOTS)
N_TRIPLET   = 4            # windows per slot; a window = 3 adjacent slices stacked as channels
IMG         = 192
CROP_MM     = 130.0
SLICE_BAND  = (0.15, 0.85)
K           = N_SLOT * N_TRIPLET

# ---- model / training ---------------------------------------------------------------------------
BACKBONE     = "resnet18"
PRETRAINED   = True        # with internet off this needs an attached weights dataset; see below
EPOCHS       = 4           # FIXED. Never chosen by looking at gold (rules.md hard rule 2)
BATCH        = 8
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-4
NUM_WORKERS  = 2

# ---- evaluation protocol (rules.md: statistical rules) ------------------------------------------
N_SEEDS   = {"smoke": 1, "full": 3, "submit": 1}[RUN_MODE]   # training seeds -> seed variance
SEEDS     = [2026, 2027, 2028][:N_SEEDS]
N_FOLDS   = 5              # evaluation folds over the 58 gold studies (NOT training folds)
EVAL_REPEATS = 5           # fold reshuffles; sigma comes from (seed x repeat x fold) cells
# Measured on a T4: with a warm cache an epoch costs ~2 s per 120 studies, so training is
# decode-bound, not compute-bound. With the prebuilt cache attached, use every weakly-labelled
# study — the old 1200 cap only ever existed to fit a decode budget the cache removes.
MAX_TRAIN_STUDIES = {"smoke": 120, "full": 10_000, "submit": 0}[RUN_MODE]
# Prebuilt tensor cache: the output of notebooks/cache-build-p1.ipynb, attached as a data source.
# Kaggle mounts a notebook's output at /kaggle/input/<notebook-slug>/ (plus our cache_<ver> subdir).
CACHE_INPUT_DIRS  = ["/kaggle/input/rsna-knee-cache-build-p1/cache_p1",
                     "/kaggle/input/rsna-knee-cache-build-p1",
                     "/kaggle/input/rsna-knee-cache-p1"]

# ---- offline weights (rules.md: internet is disabled in the rerun) ------------------------------
# Backbone weights ship as an attached dataset because the rerun cannot download anything.
# Dataset: kaggle.com/datasets/vaibhav486/timm-backbones-offline (Apache-2.0, redistributable —
# required by the winners' obligation to publish weights).
TIMM_OFFLINE_DIRS = ["/kaggle/input/timm-backbones-offline", "/kaggle/input/timm-weights"]
BACKBONE_WEIGHTS = {                      # timm model name -> file in the dataset above
    "resnet18":           "resnet18.a1_in1k.bin",
    "resnet34":           "resnet34.a1_in1k.bin",
    "tf_efficientnet_b0": "tf_efficientnet_b0.ns_jft_in1k.bin",
    "convnext_tiny":      "convnext_tiny.in12k_ft_in1k.bin",
}
CHECKPOINT_DIRS   = ["/kaggle/input/rsna-knee-baseline-v1"]   # our own trained weights, for "submit"

# ---- paths ---------------------------------------------------------------------------------------
def find_root() -> Path:
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path(".")]:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Competition data not found; set ROOT by hand.")

def find_csv(root: Path, stem: str) -> Path:
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]

ROOT  = find_root()
WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / f"cache_{PREPROC_VERSION}"          # version in the path: a stale cache cannot be reused
CACHE.mkdir(parents=True, exist_ok=True)


def discover_input(pattern: str, want_dir: bool = False) -> list[Path]:
    """Find something under /kaggle/input without guessing the mount layout.

    Kaggle has mounted attachments at BOTH /kaggle/input/<slug> and the nested
    /kaggle/input/{datasets,notebooks,competitions}/<owner>/<slug>/[version]/ — and which one you
    get is not under our control. Hardcoding either cost a wasted GPU hour once already, so search
    instead and print what was found."""
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    hits = [p for p in sorted(root.glob(pattern)) if (p.is_dir() if want_dir else p.is_file())]
    return hits


# A prebuilt cache (attached notebook output) is searched first, then our own writable one.
_cache_hits = [Path(d) for d in CACHE_INPUT_DIRS if Path(d).exists()]
_cache_hits += [d for d in discover_input(f"**/cache_{PREPROC_VERSION}", want_dir=True)
                if d not in _cache_hits and d != CACHE]
CACHE_READ = _cache_hits + [CACHE]

T_START = time.time()
def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0

def budget_check(stage: str) -> None:
    """rules.md hard rule 7: stay inside the 6.75 h working limit, loudly."""
    h = elapsed_h()
    print(f"[budget] {stage}: {h:.2f} h of {WORKING_LIMIT_H} h used ({h / KAGGLE_LIMIT_H:.0%} of the Kaggle cap)")
    if h > WORKING_LIMIT_H:
        warnings.warn(f"OVER THE WORKING BUDGET at '{stage}' — this configuration is not submittable.")

np.random.seed(SEEDS[0])
print(f"run mode   : {RUN_MODE}   seeds={SEEDS}   epochs={EPOCHS}")
print(f"data root  : {ROOT.resolve()}")
print(f"work dir   : {WORK.resolve()}")
print(f"cache      : {CACHE.name}  (preproc {PREPROC_VERSION}, labeller {LABELLER_VERSION})")
print(f"per study  : {K} windows ({N_SLOT} slots x {N_TRIPLET} triplets) at {IMG}x{IMG}")


In [ ]:
test = pd.read_csv(find_csv(ROOT, "test"))  # needed by the shared loader cell
train        = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test_series  = pd.read_csv(find_csv(ROOT, "test_series"))
for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
Y_GOLD = gold[TARGETS].values.astype(int)          # [58, 12] — the only ground truth we own

pos = pd.Series(Y_GOLD.sum(0), index=TARGETS)
print(f"gold studies: {len(gold)}   report-only: {(~gold_mask).sum()}")
print("\npositives per target among the gold studies:")
print(pos.to_string())
print(f"\nrarest target: {pos.idxmin()} with {pos.min()} positives -> "
      f"{pos.min() / N_FOLDS:.1f} expected positives per fold. "
      "This is why undefined (fold, label) cells are unavoidable and must be counted.")


In [ ]:
def normalise(text: str) -> str:
    t = unicodedata.normalize("NFKD", str(text))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t.lower())

NEG = (r"(?:no |not |without |absence of |negative for |intact |normal |unremarkable |ohne |"
       r"kein[e]?[nrms]? |unauff|sin |ausencia|geen |zonder |normale |normaal |bez |uredn|"
       r"nema |sans |pas de |absence)")

PATTERNS = {
    "ACL": r"(acl|anterior cruciate|lca|vkb|ligamento cruzado anterior|voorste kruisband|"
           r"kruisband anterior|prednj[ei] krizn|kreuzband(?:ruptur)?\s*(?:vorder)?|vorderes kreuzband)",
    "MCL": r"(mcl|medial collateral|ligamento colateral medial|innenband|mediale[nr]? kollateralband|"
           r"mediale collaterale|medijalni kolateralni)",
    "Medial Meniscus": r"(medial meniscus|menisco (?:interno|medial)|innenmeniskus|mediale meniscus|"
                       r"medijalni menisk|meniscus medialis|meniscus internus)",
    "Lateral Meniscus": r"(lateral meniscus|menisco (?:externo|lateral)|aussenmeniskus|laterale meniscus|"
                        r"lateralni menisk|meniscus lateralis)",
    "Medial OA": None, "Lateral OA": None, "PF OA": None,       # compartment co-occurrence, below
    "Effusion": r"(effusion|derrame|gelenkerguss|ergus[s]?|hydrops|izljev|epanchement|joint fluid|"
                r"gewrichtsvocht|vocht)",
    "Synovitis": r"(synovit\w*|sinovit\w*|synovialit\w*|sinovij\w*|"
                 r"synovial\w* (?:proliferation|thickening|verdikking|hypertroph\w*)|"
                 r"proliferacij\w* sinovij|pannus|synoviale? reizung)",
    "Baker's": r"(baker|popliteal cyst|quiste de baker|bakerzyste|baker-zyste|bakerova cist|kyste de baker)",
    "Contusion": r"(bone (?:marrow )?(?:contusion|bruise|oedema|edema)|contusion|knochenmarkod|kontuzij|"
                 r"botcontusie|edema oseo)",
    "Fracture": r"(fracture|fractur|fraktur|fisura osea|prijelom|breuk|fractuur|avulsion)",
}
ABNORMAL = (r"(tear|rupt|riss|scheur|rotura|lesion|desgarr|lasion|laesion|degenerativ|signal|"
            r"tearing|ruptura|insuffizienz|discontinu|abnormal)")
NEEDS_ABNORMAL = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_WORD = (r"(osteoarthrit\w*|arthros\w*|artros\w*|artroz\w*|gonarthros\w*|osteoartr\w*|chondral loss|"
           r"cartilage loss|knorpel\w*|chondropath\w*|kraakbeen\w*|hrskavic\w*|osteophyt\w*|osteofit\w*|"
           r"degenerative (?:change|veranderung)\w*|denudacij\w*)")
COMPARTMENT = {
    "Medial OA":  r"(medial\w*|mediaal|medijaln\w*|intern[oa]|innen\w*|inner)",
    "Lateral OA": r"(lateral\w*|lateraal|lateraln\w*|extern[oa]|aussen\w*|outer)",
    "PF OA":      r"(patell\w*|patel\w*|femoropatel\w*|retropatell\w*|trochlea\w*|trohlej\w*)",
}
WINDOW = 60

def _oa_label(t: str, key: str) -> float:
    pos = neg = 0
    for m in re.finditer(OA_WORD, t):
        s, e = m.span()
        ctx = t[max(0, s - WINDOW):e + WINDOW]
        if not re.search(COMPARTMENT[key], ctx):
            continue
        if key != "PF OA" and re.search(r"(?:patell|patel|trochlea|trohlej)", ctx):
            continue
        if re.search(NEG + r"[^.]{0,25}$", t[max(0, s - 45):s]):
            neg += 1
        else:
            pos += 1
    return 1.0 if pos else (0.0 if neg else np.nan)

def label_report(text: str) -> dict:
    """{finding: 1.0 | 0.0 | nan}. nan = the report does not say — NOT a zero (rules.md)."""
    t = normalise(text)
    out = {k: _oa_label(t, k) for k in COMPARTMENT}
    for key, pattern in PATTERNS.items():
        if pattern is None:
            continue
        pos = neg = 0
        for m in re.finditer(pattern, t):
            s, e = m.span()
            left, right = t[max(0, s - 45):s], t[e:e + 80]
            negated = (re.search(NEG + r"[^.]{0,25}$", left)
                       or re.search(r"^\W{0,4}(?:" + NEG + r"|ist intakt|intacto|intact)", right))
            if key in NEEDS_ABNORMAL:
                near_abnormal = re.search(ABNORMAL, right[:60]) or re.search(ABNORMAL + r"[^.]{0,30}$", left)
                if not near_abnormal:
                    if re.search(r"^\W{0,6}(?:" + NEG + r"|intact|normal)", right):
                        neg += 1
                    continue
            if negated:
                neg += 1
            else:
                pos += 1
        out[key] = 1.0 if pos else (0.0 if neg else np.nan)
    return out

weak_labels = pd.DataFrame([label_report(r) for r in train["Report"]])[TARGETS]
weak_labels.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)
print("label coverage (share of studies the report can decide):")
print((weak_labels[TARGETS].notna().mean() * 100).round(1).to_string())


## 1. How much usable training signal does each target have?

A target is learnable only in proportion to the number of report-decided cells it has — and, more
tightly, to the number of decided **positives**. Coverage alone hides this: a target can be 30%
covered and still have almost no positives to learn from.


In [ ]:
W = weak_labels[TARGETS]
rows = []
for t in TARGETS:
    col = W[t]
    n_dec = int(col.notna().sum())
    n_pos = int((col == 1).sum())
    n_neg = int((col == 0).sum())
    rows.append({"target": t, "coverage": round(n_dec / len(col), 3), "decided": n_dec,
                 "weak_pos": n_pos, "weak_neg": n_neg,
                 "pos_rate": round(n_pos / max(n_dec, 1), 3)})
signal = pd.DataFrame(rows).set_index("target")
print(signal.sort_values("weak_pos").to_string())


## 2. How corrupted is that signal?

On the 58 gold studies we know both the truth and what the labeller said. Among the *covered* cells
we can measure the two error rates directly. n is tiny, so every rate gets a Wilson 95% interval —
a point estimate from 9 positives would be storytelling.


In [ ]:
from statsmodels.stats.proportion import proportion_confint  # falls back below if unavailable
def wilson(k, n):
    if n == 0:
        return (float("nan"), float("nan"))
    try:
        lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    except Exception:
        p = k / n; z = 1.96; d = 1 + z**2 / n
        c = (p + z**2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
        lo, hi = c - h, c + h
    return round(lo, 3), round(hi, 3)

gw = weak_labels.loc[gold_mask.values, TARGETS].reset_index(drop=True)
rows = []
for j, t in enumerate(TARGETS):
    w = gw[t].values
    y = Y_GOLD[:, j]
    cov = ~np.isnan(w)
    pos_cov = cov & (y == 1)
    neg_cov = cov & (y == 0)
    fn = int(((w == 0) & pos_cov).sum()); npos = int(pos_cov.sum())
    fp = int(((w == 1) & neg_cov).sum()); nneg = int(neg_cov.sum())
    e10 = fn / npos if npos else float("nan")     # P(labelled negative | truly positive)
    e01 = fp / nneg if nneg else float("nan")     # P(labelled positive | truly negative)
    rows.append({"target": t, "gold_pos": int((y == 1).sum()), "covered": int(cov.sum()),
                 "cov_pos": npos, "cov_neg": nneg,
                 "miss_rate_e10": round(e10, 3) if npos else None, "e10_95CI": wilson(fn, npos),
                 "false_pos_e01": round(e01, 3) if nneg else None, "e01_95CI": wilson(fp, nneg)})
noise = pd.DataFrame(rows).set_index("target")
print(noise.to_string())


## 3. The ceiling question is ill-posed — and that is the finding

I proposed computing "the AUC a model would reach if it learned the weak labels perfectly." Working
through it, that number is **1.0 for every target**, which makes it useless.

Why: AUC measures *ranking*, and a model that perfectly learns `P(weak=1 | image)` ranks studies by
a monotone transform of `P(true=1 | image)` whenever the label noise is independent of the image —
symmetric or not. Independent label noise therefore does **not** cap AUC. It costs **sample
efficiency**: you need more studies to reach the same place.

So the answerable question is not "how high can we get" but **"how much data is each target
effectively training on?"** Under a standard noisy-label model, an observation carries
`(1 − e10 − e01)²` of the information of a clean one, giving an **effective sample size**

    n_eff = decided_positives × (1 − e10 − e01)²

Two real caps, which this analysis cannot measure and which must not be forgotten:
1. **Image-correlated bias** — radiologists report findings when they are *severe*. The weak label
   is then closer to "severe finding" than "finding", and the model learns the thresholded task.
   That is a genuine ceiling, and it needs gold-vs-weak disagreement cases to study, not arithmetic.
2. **A target whose evidence is not in our 12 sampled windows at all** — a geometry problem, not a
   label problem.


In [ ]:
from sklearn.metrics import roc_auc_score

model_auc = {   # baseline-v1 full run, 3-seed ensemble (Kaggle notebook v4, 2026-09-13)
    "ACL": 0.615, "MCL": 0.365, "Medial Meniscus": 0.691, "Lateral Meniscus": 0.702,
    "Medial OA": 0.727, "Lateral OA": 0.598, "PF OA": 0.537, "Effusion": 0.882,
    "Synovitis": 0.533, "Baker's": 0.779, "Contusion": 0.659, "Fracture": 0.643,
}
labeller_auc = {}
for j, t in enumerate(TARGETS):
    p = gw[t].fillna(0.5).values
    labeller_auc[t] = round(roc_auc_score(Y_GOLD[:, j], p), 3)

d = signal.join(noise[["miss_rate_e10", "false_pos_e01"]])
keep = (1 - d["miss_rate_e10"].astype(float) - d["false_pos_e01"].astype(float)).clip(lower=0)
d["signal_kept"] = keep.round(3)
d["n_eff_pos"] = (d["weak_pos"] * keep**2).round(0)
d["labeller_auc"] = pd.Series(labeller_auc)
d["model_auc"] = pd.Series(model_auc)
d["model_minus_labeller"] = (d["model_auc"] - d["labeller_auc"]).round(3)
out = d[["coverage", "weak_pos", "signal_kept", "n_eff_pos", "labeller_auc", "model_auc",
         "model_minus_labeller"]].sort_values("n_eff_pos", ascending=False)
print(out.to_string())


## 4. Does effective sample size explain the model's per-target performance?


In [ ]:
from scipy.stats import spearmanr
v = out.dropna(subset=["n_eff_pos", "model_auc"])
rho, p = spearmanr(v["n_eff_pos"], v["model_auc"])
rho_cov, p_cov = spearmanr(v["coverage"], v["model_auc"])
rho_pos, p_pos = spearmanr(v["weak_pos"], v["model_auc"])
print(f"Spearman(model AUC, n_eff_pos) = {rho:+.2f}  (p={p:.3f}, n={len(v)})")
print(f"Spearman(model AUC, coverage)  = {rho_cov:+.2f}  (p={p_cov:.3f})")
print(f"Spearman(model AUC, weak_pos)  = {rho_pos:+.2f}  (p={p_pos:.3f})")
print("\nn=12 targets, so these correlations are weak evidence: a rho this size is not significant"
      "\nunless p says so. Report the number, not a story about it.")

print("\nlabel-limited (few effective positives, model at/below labeller):")
print(out[(out["n_eff_pos"] < 200) | (out["model_minus_labeller"] < -0.05)].to_string())
print("\nmodel is already extracting more than the text rules:")
print(out[out["model_minus_labeller"] > 0.03].to_string())


## 4b. The thing the numbers are actually shouting about

`false_pos_e01` is the rate at which the labeller says **1** for a study the radiologists scored
**0**, counted only over cells the labeller claimed to decide. It runs 0.33–1.00. Together with
`pos_rate` — the share of decided cells that are positive — it says the labeller barely emits
negatives at all for several targets. Check whether "decided" effectively means "mentioned".


In [ ]:
chk = out.join(noise[["false_pos_e01", "cov_neg"]])
chk["pos_rate"] = signal["pos_rate"]
print(chk[["pos_rate", "false_pos_e01", "model_auc", "labeller_auc"]]
      .sort_values("pos_rate", ascending=False).to_string())

near_constant = chk[chk["pos_rate"] > 0.9]
print(f"\ntargets whose weak label is >90% positive among decided cells: {list(near_constant.index)}")
print(f"their mean model AUC   : {near_constant['model_auc'].mean():.3f}")
print(f"all other targets      : {chk[chk['pos_rate'] <= 0.9]['model_auc'].mean():.3f}")
print("\nA near-constant training label carries almost no gradient signal: the model cannot learn to"
      "\ndiscriminate a class it is told is always present. Compare that list with the four worst"
      "\nmodel AUCs.")

rho2, p2 = spearmanr(chk["pos_rate"], chk["model_auc"])
print(f"\nSpearman(model AUC, pos_rate) = {rho2:+.2f}  (p={p2:.3f}, n=12)")


## 4c. Pool the error rates — per-target n is far too small to read individually

`Medial OA` shows `e01 = 1.00`, but over **one** covered negative study. Pooling across targets is
the only way to get an interval worth quoting.


In [ ]:
tot_fp = tot_neg = tot_fn = tot_pos = 0
for j, t in enumerate(TARGETS):
    w = gw[t].values; y = Y_GOLD[:, j]; cov = ~np.isnan(w)
    tot_fp += int(((w == 1) & cov & (y == 0)).sum()); tot_neg += int((cov & (y == 0)).sum())
    tot_fn += int(((w == 0) & cov & (y == 1)).sum()); tot_pos += int((cov & (y == 1)).sum())

print(f"pooled over all 12 targets, counting only cells the labeller claimed to decide:")
print(f"  says POSITIVE when truth is negative : {tot_fp}/{tot_neg} = {tot_fp/tot_neg:.2f}  95% CI {wilson(tot_fp, tot_neg)}")
print(f"  says NEGATIVE when truth is positive : {tot_fn}/{tot_pos} = {tot_fn/tot_pos:.2f}  95% CI {wilson(tot_fn, tot_pos)}")

neg_share = (signal["weak_neg"] / signal["decided"]).sort_values()
print("\nshare of DECIDED cells that the labeller called negative, per target:")
print(neg_share.round(3).to_string())
print("\nA labeller that almost never says 0 is not classifying — it is detecting mentions.")


## 5. Verdict

Filled in from the output above — see `experiments/exp-20260913-03-coverage-ceiling.md`.
